# Viveka — Llama-3.2-1B RETRAIN + Inference (one file)

Retrains Llama-3.2-1B on Viveka with GRPO, then runs base + trained inference (per-action JSONs), and pushes everything to **`ddevMhrn/Llama-3.2-1B-Viveka-retrained`** — a NEW repo, so the original `ddevMhrn/Llama-3.2-1B-Viveka` is left untouched.

**Why retrain 1B:** the original run shipped only `.log` SUMMARY blocks. To tell *reward hacking* (model exploits the task_progress component, sacrifices caution) from plain *capacity failure* (too small to learn, degrades uniformly), we need the per-action `components` + `trajectory` JSONs. This notebook produces them.

**The reward-hacking signature to look for** (after running `eval/capability_report.py --prefix llama1b_retrained` locally on the pushed JSONs):
- `task_progress` UP, `confirmation_appropriate` / `reversibility_correct` DOWN → reward hacking
- everything DOWN uniformly → capacity failure

## Fixes baked in (the reasoning, so future-you knows why)

1. **Install order** — `pip install -e "."` pulls scikit-learn/scipy from pyproject and realigns Kaggle's skewed base numpy (else `_blas_supports_fpe` / `_center` import errors).
2. **`weave`** — TRL ≥ 0.16 imports it inside `grpo_trainer`; training subprocess needs the real package.
3. **`torchao` uninstall** — peft ≥ 0.14's `is_torchao_available()` *raises* on torchao < 0.16 (Kaggle ships 0.10), which aborts `PeftModel.from_pretrained` during inference.
4. **EOS fix** — already in `train.py` (discovers `<|eot_id|>` etc. and pins `generation_kwargs`); no action needed here, Llama-3.2 has the multi-EOS that the fix handles.
5. **Inference base = Unsloth 4-bit mirror** (`unsloth/Llama-3.2-1B-Instruct-bnb-4bit`) — `meta-llama` is gated and `inference.py` loads via plain `transformers` (no bnb config), so the canonical id would 403. Same weights training used.

## Prereqs

1. **Settings:** Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`
2. **Add-ons → Secrets:** `HF_TOKEN` (WRITE scope — pushes to `ddevMhrn/Llama-3.2-1B-Viveka-retrained`)
3. Run cells in order. Don't run full training (Step 6) until smoke (Step 5) is clean.

## Time budget

~45-90 min training (1B is fast) + ~30-45 min inference (4 passes, per-tier 5) + push. Well under the 12 h cap.


In [ ]:
# Step 1: GPU check + clone HF Space + tokens ───────────────────────
import os
from kaggle_secrets import UserSecretsClient

TRAIN_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
EVAL_MODEL  = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
RUN_NAME    = "llama1b_retrained_v1"
REPO_ID     = "ddevMhrn/Llama-3.2-1B-Viveka-retrained"
LOG_PREFIX  = "llama1b_retrained"
RUN_DIR     = f"/kaggle/working/runs/{RUN_NAME}"

print(f"Train model:  {TRAIN_MODEL}  (Unsloth redirects to open 4-bit mirror)")
print(f"Eval base:    {EVAL_MODEL}")
print(f"Run dir:      {RUN_DIR}")
print(f"Push to:      {REPO_ID}  (NEW repo — original untouched)")

os.chdir("/"); os.chdir("/kaggle/working")
%cd /kaggle/working
!nvidia-smi | head -20

!rm -rf /kaggle/working/viveka-env
!git lfs install --skip-repo 2>/dev/null || true
!git clone https://huggingface.co/spaces/ddevMhrn/viveka-env /kaggle/working/viveka-env
%cd /kaggle/working/viveka-env

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("\nHF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
!git log --oneline -3


## Step 2: Install — full training stack (superset of inference deps)

Training needs trl + unsloth + weave; inference needs peft + bitsandbytes (Unsloth pulls these). The `pip install -e "."` first is what realigns numpy. torchao is removed at the end for the inference PEFT path.


In [ ]:
# Step 2: Install all deps (training + inference) ───────────────────
# pip install -e "." FIRST — pulls scikit-learn/scipy from pyproject and
# realigns Kaggle's skewed base numpy. (Skipping this is what broke the AQI
# notebooks with _blas_supports_fpe / _center import errors.)
!pip install -q -e ".[train]"

# openenv-core / fastmcp pinned pair
!pip install --upgrade --force-reinstall --no-deps "openenv-core==0.2.2" "fastmcp==3.1.1"
!pip install -q -U "mcp"
!pip install -q -U "uncalled-for"

# TRL (GRPOConfig) + transformers/peft/bnb/accelerate
!pip install -q -U "trl>=0.13.0"
!pip install -q mergekit

# Unsloth (4-bit QLoRA training) + bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U bitsandbytes

# weave — TRL >= 0.16 imports it inside grpo_trainer; subprocess needs the real pkg
!pip install -q weave

# torchao uninstall — peft >= 0.14's is_torchao_available() RAISES on torchao
# < 0.16 (Kaggle ships 0.10), aborting PeftModel.from_pretrained at inference.
!pip uninstall -y torchao 2>&1 | tail -2

# HF Hub for the push
!pip install -q -U "huggingface_hub[cli]"

print("\n=== versions ===")
!pip show openenv-core fastmcp trl transformers peft unsloth bitsandbytes huggingface-hub 2>&1 | grep -E "^(Name|Version)" 


In [ ]:
# Step 3: Verify imports (training + inference paths) ───────────────
import importlib, importlib.util, sys, types

# Stubs for TRL's broken transitive imports (must precede trl import)
class _DummyModule(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith("__"): raise AttributeError(name)
        return type(name, (), {})
def _install_stub(modname, dummy=False):
    if modname in sys.modules: return
    m = _DummyModule(modname) if dummy else types.ModuleType(modname)
    m.__spec__ = importlib.util.spec_from_loader(modname, None)
    sys.modules[modname] = m
try:
    import llm_blender  # noqa: F401
except Exception:
    _install_stub("llm_blender"); sys.modules["llm_blender"].Blender = type("Blender", (), {})
for _n in ["mergekit","mergekit.merge_methods","mergekit.io","mergekit.config",
           "mergekit.architecture","mergekit.options","mergekit.merge","mergekit.plan","mergekit.graph"]:
    _install_stub(_n, dummy=True)
print("shims applied")

# Unsloth FIRST so its patches apply before transformers/trl/peft load
ok = False
try:
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        from unsloth import FastLanguageModel  # noqa: F401
    print("\u2705 unsloth.FastLanguageModel"); ok = True
except Exception as e:
    print(f"\u274c unsloth: {type(e).__name__}: {e}")

def test(m):
    try: importlib.import_module(m); print(f"\u2705 {m}"); return True
    except Exception as e: print(f"\u274c {m}: {type(e).__name__}: {e}"); return False
ok &= test("trl")
ok &= test("openenv.core.env_server.types")
try:
    from trl import GRPOConfig, GRPOTrainer  # noqa: F401
    print("\u2705 trl.GRPOConfig + GRPOTrainer")
except Exception as e:
    print(f"\u274c GRPO: {type(e).__name__}: {e}"); ok = False
try:
    sys.path.insert(0, "/kaggle/working/viveka-env")
    from viveka.server.environment import VivekaEnvironment  # noqa: F401
    print("\u2705 viveka.server.environment");
except Exception as e:
    print(f"\u274c viveka env: {type(e).__name__}: {e}"); ok = False
print(f"\n{'\u2705 ALL CLEAN' if ok else '\u274c FIX BEFORE PROCEEDING'}")


In [ ]:
# Step 4: Dry-run (no GPU touch) ────────────────────────────────────
!python train.py --dry-run --model meta-llama/Llama-3.2-1B-Instruct


## Smoke run (10 episodes)

Watch for `[fix] generation_kwargs.eos_token_id list = [...]` (EOS workaround armed), training step logs, no `[NaNGuard]` halt, final `[smoke] terminal reward=`. Llama-3.2 has multi-EOS — the in-repo fix handles it.


In [ ]:
# Step 5: Smoke run (10 episodes) ───────────────────────────────────
!python train.py --smoke \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --output-dir /kaggle/working/runs/smoke \
    --no-wandb


## Full training — 200 episodes, tier mix 1:0.4 / 2:0.4 / 4:0.2

Same config as the original Llama-1B run, so the retrained model is comparable. Trained LoRA lands at `/kaggle/working/runs/llama1b_retrained_v1/lora`. Watch the reward trajectory — for the 1B we expect the known collapse pattern (training reward may rise while the model drifts toward execute-spam).


In [ ]:
# Step 6: Full training (200 episodes) ──────────────────────────────
!python train.py \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --episodes 200 \
    --output-dir $RUN_DIR \
    --tier-mix "1:0.4,2:0.4,4:0.2" \
    --no-wandb


In [ ]:
# Step 7: Reward curve from training log ────────────────────────────
!python eval/reward_curve.py \
    --training-log $RUN_DIR/training_log.jsonl \
    --baseline-json eval/results/baseline_random.json \
    --output-png $RUN_DIR/reward_curve.png \
    --smooth-window 10 \
    --title "GRPO Training — Llama-3.2-1B retrained (200 episodes)" 2>&1 | tail -5
!ls -la $RUN_DIR/


## Inference — base vs trained, T1+T2 then T3+T4

Four `inference.py` passes producing per-action JSONs (`components` + `trajectory`). `--per-tier 5` matches the original Llama-1B convention so the retrained numbers are directly comparable. Base loads from the Unsloth 4-bit mirror; trained layers the freshly-trained local LoRA on top.


In [ ]:
# Step 8: Inference — FROZEN base on T1+T2 ─────────────────────────
out_json = f"{RUN_DIR}/llama1b_retrained_base_t12.json"
out_log  = f"{RUN_DIR}/llama1b_retrained_base_t12.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --tier-mix 1,2 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 9: Inference — FROZEN base on T3+T4 ─────────────────────────
out_json = f"{RUN_DIR}/llama1b_retrained_base_t34.json"
out_log  = f"{RUN_DIR}/llama1b_retrained_base_t34.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --tier-mix 3,4 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 10: Inference — TRAINED (base + local LoRA) on T1+T2 ────────
out_json = f"{RUN_DIR}/llama1b_retrained_train_t12.json"
out_log  = f"{RUN_DIR}/llama1b_retrained_train_t12.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --adapter $RUN_DIR/lora \
    --tier-mix 1,2 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 11: Inference — TRAINED on T3+T4 (the showcase tier) ────────
out_json = f"{RUN_DIR}/llama1b_retrained_train_t34.json"
out_log  = f"{RUN_DIR}/llama1b_retrained_train_t34.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --adapter $RUN_DIR/lora \
    --tier-mix 3,4 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log

print("\n=== base vs trained summaries ===")
log_glob = f"{RUN_DIR}/llama1b_retrained_*.log"
!grep -A 7 "SUMMARY" $log_glob


## Push to HF Hub: `ddevMhrn/Llama-3.2-1B-Viveka-retrained` (NEW repo)

Uploads LoRA + training_log + reward_curve + 4 inference JSON/LOG files to the **retrained** repo. The original `ddevMhrn/Llama-3.2-1B-Viveka` is not touched.


In [ ]:
# Step 12: Push LoRA + artifacts to the RETRAINED repo ──────────────
import os, shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo

RUN_DIR_P = Path(RUN_DIR)
LORA_DIR = RUN_DIR_P / "lora"
assert LORA_DIR.exists(), f"missing LoRA dir: {LORA_DIR}"

for fname in [
    "training_log.jsonl", "reward_curve.png",
    f"llama1b_retrained_base_t12.log",  f"llama1b_retrained_base_t12.json",
    f"llama1b_retrained_base_t34.log",  f"llama1b_retrained_base_t34.json",
    f"llama1b_retrained_train_t12.log", f"llama1b_retrained_train_t12.json",
    f"llama1b_retrained_train_t34.log", f"llama1b_retrained_train_t34.json",
]:
    src = RUN_DIR_P / fname
    if src.exists():
        shutil.copy(src, LORA_DIR / fname); print(f"  bundled {fname}")

card = """---
library_name: peft
base_model: meta-llama/Llama-3.2-1B-Instruct
license: apache-2.0
tags: [viveka, grpo, reversibility, calibrated-confidence, indic-dpi, openenv]
---

# Llama-3.2-1B-Viveka (retrained)

Fresh retrain of Llama-3.2-1B on the [Viveka OpenEnv](https://huggingface.co/spaces/ddevMhrn/viveka-env), produced to obtain per-action inference JSONs (components + trajectory) for capacity-failure-vs-reward-hacking analysis. 200 episodes, tier mix 1:0.4 / 2:0.4 / 4:0.2. Separate from the original repo so prior artifacts are preserved.

**Base model:** `meta-llama/Llama-3.2-1B-Instruct` (trained via Unsloth's open 4-bit mirror).

See [github.com/DevMhrn/viveka-env](https://github.com/DevMhrn/viveka-env).
"""
(LORA_DIR / "README.md").write_text(card)

create_repo(REPO_ID, repo_type="model", exist_ok=True, private=False, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(folder_path=str(LORA_DIR), repo_id=REPO_ID, repo_type="model",
                     token=os.environ["HF_TOKEN"],
                     commit_message="retrain Llama-3.2-1B + base/trained inference (per-action JSONs)")
print(f"\n\u2705 pushed to https://huggingface.co/{REPO_ID}")
